# 04 - Gold Layer & Business Analytics

O objetivo da camada Gold é transformar os dados limpos da Silver em informação analítica pronta para responder a questões de negócio. Nesta fase são efetuados os JOINs entre as tabelas principais do FAERS (`DEMO`, `DRUG`, `REAC` e `OUTC`), criando uma visão consolidada dos casos reportados.



⚠️ No FAERS, um caso (primaryid) pode ter vários medicamentos (`DRUG`), várias reações (`REAC`) e vários desfechos (`OUTC`). Fazer um JOIN direto a todas as tabelas vai multiplicar as linhas (ex: 1 caso com 2 medicamentos e 3 reações = 6 linhas). Isto é o chamado fan-out.
Para garantir que as contagens ficam corretas, vamos usar sempre countDistinct("primaryid") em vez de um simples count() quando quisermos contar o número real de casos.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast

# 1. Definir o caminho onde as tabelas Silver foram guardadas
silver_delta_path = "/Volumes/main/default/faers_data/delta/silver"

# 2. Carregar as tabelas Delta para o dicionário silver_dfs
tables = ["demo", "drug", "reac", "outc"]
silver_dfs = {}

for table in tables:
    silver_dfs[table] = spark.read.format("delta").load(f"{silver_delta_path}/{table}")
    print(f"✅ Tabela {table.upper()} carregada com sucesso.")

# 3. Criar a Gold Base Table

df_gold_base = (
    silver_dfs["demo"]
    .join(silver_dfs["drug"], on=["primaryid", "caseid"], how="inner")
    .join(silver_dfs["reac"], on=["primaryid", "caseid"], how="left")
    .join(F.broadcast(silver_dfs["outc"]), on=["primaryid", "caseid"], how="left")
    .withColumn("_gold_created_at", F.current_timestamp())
)

display(df_gold_base.limit(5))

# 4. # Mostrar plano de execução para comprovar a otimização
df_gold_base.explain(True)

# 5. Registar como View SQL

df_gold_base.createOrReplaceTempView("gold_base")

print("✅ View SQL 'gold_base' criada.")

A otimização Broadcast Join foi aplicada durante a junção da tabela `OUTC` com o conjunto de dados da camada Gold. Como a tabela `OUTC` é relativamente pequena quando comparada com as restantes tabelas utilizadas no processo, o Spark distribuiu uma cópia desta tabela por todos os executores, evitando a necessidade de realizar uma operação completa de shuffle dos dados.

O plano de execução confirma que a otimização foi aplicada através da presença dos operadores PhotonBroadcastHashJoin, BuildRight e EXECUTOR_BROADCAST. Esta abordagem reduz a movimentação de dados pela rede, melhora a eficiência das operações de junção e permite obter os mesmos resultados analíticos com um desempenho superior.

# Q1 - Top 10 medicamentos mais reportados e as reações adversas mais associadas a cada um

Foram identificados os 10 medicamentos mais frequentemente reportados na base de dados `FAERS`. Para cada medicamento foram calculadas as três reações adversas mais frequentemente associadas, utilizando countDistinct(`primaryid`) para evitar duplicações resultantes do fan-out criado pelos joins entre as tabelas `DEMO`, `DRUG`, `REAC` e `OUTC`. O ranking das reações foi obtido através de Window Functions (`row_number`).

In [0]:
# 1. Encontrar os 10 medicamentos mais reportados
top_10_drugs = (
    df_gold_base
    .groupBy("drugname")
    .agg(
        F.countDistinct("primaryid").alias("total_casos") # Evita contagens inflacionadas pelo fan-out
    )  
    .orderBy(F.desc("total_casos"))
    .limit(10)
)

# 2. Contar reações adversas associadas a esses medicamentos
reactions_per_drug = (
    df_gold_base
    .join(top_10_drugs.select("drugname"), on="drugname", how="inner")
    .filter(
        F.col("pt").isNotNull() &
        (F.col("pt") != "UNK")
    ) # Ignoramos reações nulas
    .groupBy("drugname", "pt")
    .agg(
        F.countDistinct("primaryid").alias("reaction_count") # Evita duplicações causadas pelos joins
    )  
)

# 3. Ranking das reações por medicamento
w_q1 = (
    Window
    .partitionBy("drugname")
    .orderBy(
        F.desc("reaction_count"),
        F.asc("pt")
    )
)

# 4. Extrair as 3 reações mais frequentes de cada medicamento
q1_result = (
    reactions_per_drug
    .withColumn(
        "rank",
        F.row_number().over(w_q1)
    )
    .filter(F.col("rank") <= 3)
    .orderBy(
        "drugname",
        "rank"
    )
)

display(q1_result)

In [0]:
%sql

-- 1. Identificar os 10 medicamentos com mais casos reportados
WITH top_10_drugs AS (
    SELECT
        drugname,
        COUNT(DISTINCT primaryid) AS total_cases
    FROM gold_base
    GROUP BY drugname
    ORDER BY total_cases DESC
    LIMIT 10
),

-- 2. Contar quantas vezes cada reação foi associada a cada um dos 10 medicamentos mais reportados
reaction_counts AS (
    SELECT
        g.drugname,
        g.pt,
        COUNT(DISTINCT g.primaryid) AS reaction_count
    FROM gold_base g
    INNER JOIN top_10_drugs t
        ON g.drugname = t.drugname
    WHERE g.pt IS NOT NULL
    GROUP BY g.drugname, g.pt
),

-- 3. Ordenar as reações por frequência dentro de cada medicamento e atribuir uma posição (ranking)
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER(
               PARTITION BY drugname
               ORDER BY reaction_count DESC
           ) AS rn
    FROM reaction_counts
)

-- 4. Selecionar apenas as 3 reações mais frequentes para cada um dos 10 medicamentos
SELECT *
FROM ranked
WHERE rn <= 3
ORDER BY drugname, rn;

Os resultados mostram que alguns medicamentos apresentam padrões de reações adversas bastante distintos. O `ACETAMINOPHEN` surge frequentemente associado a OFF LABEL USE, FATIGUE e HEADACHE, sendo esta última uma das reações adversas clínicas mais comuns observadas. De forma semelhante, o `ASPIRIN` apresenta uma elevada associação a FATIGUE e DIARRHOEA, além de numerosos registos de OFF LABEL USE.

O `ATORVASTATIN` evidencia um perfil semelhante ao do `ASPIRIN`, destacando-se as reações FATIGUE e DIARRHOEA. Já o `DEXAMETHASONE` apresenta uma distribuição diferente, surgindo frequentemente associado a DRUG INEFFECTIVE, o que poderá indicar situações em que o tratamento não produziu o efeito terapêutico esperado.

O `DUPIXENT` destaca-se por estar fortemente associado a reações dermatológicas, nomeadamente PRURITUS e RASH, resultados consistentes com o seu contexto de utilização em doenças inflamatórias e dermatológicas.

Importa referir que algumas das ocorrências mais frequentes, como OFF LABEL USE ou PRODUCT USE IN UNAPPROVED INDICATION, não correspondem necessariamente a reações adversas clínicas, mas sim a circunstâncias relacionadas com a utilização do medicamento. Ainda assim, estas informações são relevantes do ponto de vista da farmacovigilância, uma vez que ajudam a compreender os contextos em que os medicamentos são utilizados e reportados.

# Q2 - Distribuição de eventos adversos por faixa etária

Os eventos adversos foram distribuídos por três grupos etários (`Pediatric`, `Adult` e `Elderly`). Para evitar sobrecontagem causada pelo fan-out dos joins, utilizou-se countDistinct(`primaryid`). A percentagem de cada grupo foi calculada através de uma Window Function aplicada ao total global de eventos.

In [0]:
# Utilizar a coluna age_grp_cleaned normalizada na camada Silver
q2_result = (
    df_gold_base
    .filter( 
        (F.col("age_grp_cleaned").isNotNull()) & # Remove valores nulos e registos classificados como "UNK" (idade desconhecida)
        (F.col("age_grp_cleaned") != "UNK")
    )
    .groupBy("age_grp_cleaned")  # Agrupa os dados por grupo etário
    .agg(
        F.countDistinct("primaryid").alias("total_events") # Conta o número de eventos únicos (primaryid) em cada grupo etário
    )
    .withColumn(
        "percentage",  # Calcula a percentagem que cada grupo etário representa relativamente ao total de eventos analisados
        F.round(
            (
                F.col("total_events")
                / F.sum("total_events").over(Window.partitionBy())
            ) * 100,
            2
        )
    )
    .orderBy(F.desc("total_events")) # Ordena os resultados por número de eventos, do maior para o menor
)

display(q2_result)

In [0]:
%sql

-- 1. Contar eventos adversos por grupo etário já normalizado na Silver
WITH age_groups AS (

    SELECT
        age_grp_cleaned AS age_group,

        -- Contar casos únicos para evitar sobrecontagem
        COUNT(DISTINCT primaryid) AS total_events

    FROM gold_base

    -- Excluir idades desconhecidas
    WHERE age_grp_cleaned <> 'UNK'

    GROUP BY age_grp_cleaned
)

-- 2. Calcular a percentagem de cada grupo relativamente ao total
SELECT
    age_group,
    total_events,

    ROUND(
        total_events * 100.0 /
        SUM(total_events) OVER (),
        2
    ) AS percentage

FROM age_groups

-- Ordenar do grupo com mais eventos para o grupo com menos eventos
ORDER BY total_events DESC;

A análise da distribuição dos eventos adversos por faixa etária revelou que os `ADULT` (18–64 anos) constituem o grupo mais representado, concentrando 53,68% dos casos reportados. Os `ELDERLY` (65+ anos) representam 39,69% dos eventos, evidenciando uma elevada carga de notificações nesta população, possivelmente associada à maior prevalência de doenças crónicas, polimedicação e maior exposição a tratamentos farmacológicos.

Os grupos pediátricos apresentam uma expressão significativamente menor. Os `ADOLESCENT` correspondem a 2,95% dos casos, as `CHILD` a 2,86%, os `INFANT` a 0,56% e os `NEONATE` a apenas 0,26% dos eventos reportados.

Globalmente, os resultados demonstram que mais de 93% dos eventos adversos registados na base de dados FAERS ocorrem em populações adultas e idosas. Esta distribuição sugere que estes grupos concentram a maior exposição a medicamentos e, consequentemente, o maior risco de ocorrência e reporte de eventos adversos, reforçando a importância da monitorização contínua da segurança farmacológica nestas faixas etárias.

# Q3 - Pares Medicamento-Reação com maior taxa de mortalidade


   
Para identificar os medicamentos associados às maiores taxas de mortalidade, foram considerados todos os casos reportados na base de dados `FAERS` com um medicamento válido associado. Para cada medicamento, foi calculado o número total de casos reportados e o número de casos cujo outcome incluía morte (`DE`).

A taxa de mortalidade foi calculada como a razão entre o número de casos com outcome de morte e o total de casos associados a cada medicamento, sendo expressa em percentagem. Para garantir a precisão dos resultados e evitar contagens duplicadas resultantes dos joins efetuados na construção da camada Gold, foi utilizada a contagem distinta de casos através do identificador `primaryid`.

Foram excluídos da análise os medicamentos com taxa de mortalidade igual a 100%, uma vez que estes resultados estão frequentemente associados a um número muito reduzido de notificações, podendo introduzir enviesamentos e interpretações pouco representativas. Os medicamentos foram posteriormente ordenados por taxa de mortalidade de forma decrescente, permitindo identificar aqueles que apresentam maior proporção de casos fatais reportados.

In [0]:

# Calcular a taxa de mortalidade por medicamento
q3_result = (
    df_gold_base

    # Remove registos sem medicamento associado
    .filter(F.col("drugname").isNotNull())

    # Agrupa os dados por medicamento
    .groupBy("drugname")

    # Calcula o total de casos e o total de casos com outcome de morte
    .agg(
        # Conta o número de casos únicos associados a cada medicamento
        F.countDistinct("primaryid").alias("total_cases"),

        # Conta o número de casos únicos cujo outcome foi morte (DE = Death)
        F.countDistinct(
            F.when(
                F.col("outc_cod") == "DE",
                F.col("primaryid")
            )
        ).alias("death_cases")
    )

    # Calcula a taxa de mortalidade para cada medicamento
    .withColumn(
        "mortality_rate_pct",
        F.round(
            (
                F.col("death_cases")
                / F.col("total_cases")
            ) * 100,
            2
        )
    )

    # Remove medicamentos com taxa de mortalidade de 100% (normalmente associados a poucos casos e potencialmente enviesados)
    .filter(F.col("mortality_rate_pct") < 100)

    # Ordena os resultados pela taxa de mortalidade, da maior para a menor
    .orderBy(F.desc("mortality_rate_pct"))
)

# Apresenta os 1000 medicamentos com maior taxa de mortalidade
display(q3_result.limit(1000))


In [0]:
# Teste de validação da lógica da Q3 utilizando apenas o medicamento COCAINE
# O objetivo é confirmar que o cálculo da taxa de mortalidade está correto antes de o aplicar a todos os medicamentos da base de dados

(df_gold_base
    .filter(F.upper(F.col("drugname")) == "DEXTROMETHORPHANHYDROBROMIDEQUINIDINESULFATE")
    .groupBy("drugname")
    .agg(
        F.countDistinct("primaryid").alias("total_cases"),
        F.countDistinct(
            F.when(F.col("outc_cod") == "DE", F.col("primaryid"))
        ).alias("death_cases")
    )
    .withColumn(
        "mortality_rate_pct",
        F.round(
            F.col("death_cases") / F.col("total_cases") * 100,
            2
        )
    )
).display()

In [0]:
# Análise dos outcomes reportados para o medicamento COCAINE
# O objetivo é verificar a distribuição dos diferentes outcomes e confirmar o número de casos classificados como morte (DE), utilizado posteriormente no cálculo da taxa de mortalidade

(
    df_gold_base
    .filter(F.upper(F.col("drugname")) == "COCAINE")
    .groupBy("outc_cod")
    .agg(
        F.countDistinct("primaryid").alias("num_cases")
    )
    .orderBy(F.desc("num_cases"))
).display()

In [0]:
# Validação detalhada dos outcomes por caso para o medicamento COCAINE
# O objetivo é verificar quais os outcomes associados a cada caso individual (primaryid) e confirmar que um mesmo caso pode possuir múltiplos outcomes reportados no sistema FAERS

(
    df_gold_base
    .filter(F.upper(F.col("drugname")) == "COCAINE")
    .groupBy("primaryid")
    .agg(
        F.collect_set("outc_cod").alias("outcomes")
    )
    .display()
)

In [0]:
%sql
    
-- Taxa de mortalidade por medicamento

SELECT
    drugname,

    -- Conta o número total de casos únicos associados a cada medicamento
    COUNT(DISTINCT primaryid) AS total_cases,

    -- Conta o número de casos únicos cujo outcome foi morte (DE = Death)
    COUNT(DISTINCT CASE
        WHEN outc_cod = 'DE' THEN primaryid
    END) AS death_cases,

    -- Calcula a taxa de mortalidade (%)
    ROUND(
        (
            COUNT(DISTINCT CASE
                WHEN outc_cod = 'DE' THEN primaryid
            END) * 100.0
        ) / COUNT(DISTINCT primaryid),
        2
    ) AS mortality_rate_pct

FROM gold_base

-- Remove registos sem medicamento associado
WHERE drugname IS NOT NULL

-- Agrupa os dados por medicamento
GROUP BY drugname

-- Remove medicamentos com taxa de mortalidade de 100%
HAVING ROUND(
    (
        COUNT(DISTINCT CASE
            WHEN outc_cod = 'DE' THEN primaryid
        END) * 100.0
    ) / COUNT(DISTINCT primaryid),
    2
) < 100

-- Ordena os resultados pela taxa de mortalidade, da maior para a menor
ORDER BY mortality_rate_pct DESC

-- Apresenta os 1000 medicamentos com maior taxa de mortalidade
LIMIT 1000;

   
A análise dos medicamentos com maior taxa de mortalidade permitiu identificar substâncias que apresentam uma elevada proporção de casos fatais entre os eventos reportados no sistema FAERS. A taxa de mortalidade foi calculada para cada medicamento com base na relação entre o número de casos com outcome de morte (`DE`) e o número total de casos associados ao respetivo medicamento.

Os resultados mostram que vários medicamentos apresentam taxas de mortalidade superiores a 90%, destacando-se substâncias como `DEXTROSEHEPARINSODIUM`, `ISUPREL`, `MALTOSE`, `OTILIMAB` e `HERBALSPAULLINIACUPANASEED`. Em muitos destes casos, observa-se uma proporção muito elevada de outcomes fatais relativamente ao número total de notificações registadas.

Contudo, a interpretação destes resultados deve ser feita com cautela. Alguns medicamentos apresentam um número relativamente reduzido de casos reportados, o que pode amplificar a taxa de mortalidade observada. Por outro lado, determinados medicamentos são utilizados em contextos clínicos particularmente graves ou em doentes com elevado risco de mortalidade, pelo que a frequência de outcomes fatais pode refletir a gravidade da condição tratada e não necessariamente um efeito direto do medicamento.

É importante salientar que os dados do FAERS resultam de notificações espontâneas de eventos adversos e, por isso, não permitem estabelecer relações de causalidade. Ainda assim, a identificação de medicamentos associados a elevadas proporções de outcomes fatais constitui um indicador relevante para atividades de farmacovigilância, podendo apoiar processos de monitorização, investigação e avaliação de segurança.

# Q4 - Padrões sazonais no reporte de eventos (por mês)

Para identificar padrões sazonais no reporte de eventos adversos, os casos foram agrupados por estação do ano com base na data do evento (`event_dt`). A contagem foi realizada utilizando countDistinct(`primaryid`) para evitar sobrecontagem decorrente dos joins da Gold Table. Foram ainda calculadas as percentagens relativas de cada estação para facilitar a comparação entre períodos do ano.

In [0]:
# Distribuição sazonal dos relatórios de eventos adversos
# O objetivo é analisar se existem variações no número de notificações ao longo do ano, agrupando os casos por mês e estação do ano

q4_result = (

    df_gold_base

    # Remove registos sem data do evento
    .filter(F.col("event_dt").isNotNull())

    # Extrai o mês da data do evento
    .withColumn("event_month", F.month("event_dt"))

    # Classifica cada mês na respetiva estação do ano
    .withColumn(
        "season",
        F.when(F.col("event_month").isin(12, 1, 2), "Winter")
         .when(F.col("event_month").isin(3, 4, 5), "Spring")
         .when(F.col("event_month").isin(6, 7, 8), "Summer")
         .otherwise("Autumn")
    )

    # Agrupa os dados por mês e estação do ano
    .groupBy("event_month", "season")

    # Conta o número de casos únicos reportados
    .agg(
        F.countDistinct("primaryid").alias("total_reports")
    )

    # Ordena os resultados cronologicamente por mês
    .orderBy("event_month")
)

# Apresenta a distribuição dos relatórios por mês e estação do ano
display(q4_result)

In [0]:
%sql
-- 1. Criar uma tabela temporária com o mês e a estação do ano associadas a cada evento adverso
WITH seasonal_reports AS (

    SELECT
        MONTH(event_dt) AS event_month, -- Extrair o mês da data do evento

        CASE -- Classificar cada evento numa estação do ano
            WHEN MONTH(event_dt) IN (12, 1, 2) THEN 'Winter'
            WHEN MONTH(event_dt) IN (3, 4, 5) THEN 'Spring'
            WHEN MONTH(event_dt) IN (6, 7, 8) THEN 'Summer'
            ELSE 'Autumn'
        END AS season,

        primaryid

    FROM gold_base

    WHERE event_dt IS NOT NULL  -- Excluir registos sem data do evento
)

SELECT
    event_month,
    season,
    COUNT(DISTINCT primaryid) AS total_reports  -- Contagem de casos únicos

FROM seasonal_reports

GROUP BY event_month, season

ORDER BY event_month; -- Ordenar pelos meses do ano

A análise da distribuição temporal dos reportes permitiu identificar variações ao longo dos meses e das estações do ano. Observa-se que os meses de setembro, outubro e novembro apresentam os maiores volumes de reportes, correspondendo ao período de Outono, enquanto os meses de fevereiro, abril e maio registam valores relativamente inferiores.

O mês com o maior número de reportes foi outubro, com 85.985 casos, seguido de novembro (81.512) e setembro (75.858). Em contraste, fevereiro apresentou o menor número de reportes, com apenas 56.910 casos.

Quando os resultados são analisados por estação do ano, verifica-se que o Inverno com cerca de 350.744 reportes impulsionado sobretudo pelo elevado número de casos registados em janeiro . Seguem-se o Outono totalizando aproximadamente 243.355 reportes, e o Verão com aproximadamente 190.160 reportes. A Primavera apresenta o menor volume global de notificações, com cerca de 180.508 reportes.

Estes resultados sugerem a existência de alguma sazonalidade nos reportes de eventos adversos. No entanto, as diferenças observadas não devem ser interpretadas como evidência direta de maior risco associado aos medicamentos em determinadas épocas do ano. Fatores como padrões de utilização dos medicamentos, campanhas de vacinação, prevalência sazonal de determinadas doenças e alterações nos comportamentos de notificação podem influenciar o número de casos reportados.

De forma global, a análise demonstra que os reportes de eventos adversos não se distribuem uniformemente ao longo do ano, evidenciando períodos de maior atividade de farmacovigilância, particularmente durante os meses de Outono e início do Inverno.

# Q5 - Duração média da terapêutica por medicamento

In [0]:
print("""
Não foi possível calcular a duração média da terapêutica.
A tabela THER, necessária para obter datas de início e fim do tratamento, não foi incluída no pipeline Silver/Gold.
""")

# Q6 - Diferença de Outcomes nos Top 5 Medicamentos

Foram comparados os outcomes clínicos dos cinco medicamentos mais frequentemente reportados. Para evitar duplicações provocadas pelo fan-out dos joins, utilizou-se countDistinct(`primaryid`). Os resultados permitem identificar diferenças no perfil de gravidade dos eventos adversos, nomeadamente mortalidade (`DE`), hospitalização (`HO`) e situações life-threatening (`LT`), entre os medicamentos mais reportados na base FAERS.

In [0]:
# 1. Top 5 medicamentos mais reportados
top_5_drugs = (
    df_gold_base
    .groupBy("drugname")
    .agg(
        F.countDistinct("primaryid").alias("total_cases")
    )
    .orderBy(F.desc("total_cases"))
    .limit(5)
)

top_5_list = [row["drugname"] for row in top_5_drugs.collect()]

# 2. Comparar outcomes entre os Top 5 medicamentos
q6_result = (
    df_gold_base

    .filter(
        F.col("drugname").isin(top_5_list)
        & F.col("outc_cod").isNotNull()
        & (F.col("outc_cod") != "UNK")
    )

    .groupBy("drugname")

    .pivot(
        "outc_cod",
        ["DE", "HO", "LT", "DS", "CA", "RI", "OT"]
    )

    .agg(
        F.countDistinct("primaryid")
    )

    .fillna(0)

    .orderBy("drugname")
)

display(q6_result)

In [0]:
%sql
-- 1. Identificar os 5 medicamentos com maior número de casos reportados
WITH top_5_drugs AS (

    SELECT
        drugname,
        COUNT(DISTINCT primaryid) AS total_cases -- Contagem de casos únicos por medicamento

    FROM gold_base

    GROUP BY drugname

    ORDER BY total_cases DESC -- Ordenar por número de casos (maior para menor)

    LIMIT 5 -- Selecionar apenas os 5 medicamentos mais reportados
)
-- 2. Selecionar os outcomes associados aos medicamentos do Top 5
SELECT *

FROM (

    SELECT
        g.drugname,
        g.outc_cod,
        g.primaryid

    FROM gold_base g

    INNER JOIN top_5_drugs t
        ON g.drugname = t.drugname

-- Excluir outcomes nulos ou desconhecidos
    WHERE g.outc_cod IS NOT NULL  
      AND g.outc_cod <> 'UNK'
)

-- 3. Transformar os outcomes em colunas
PIVOT (

    COUNT(DISTINCT primaryid)

    FOR outc_cod IN (
        'DE',
        'HO',
        'LT',
        'DS',
        'CA',
        'RI',
        'OT'
    )

)
-- 4. Ordenar alfabeticamente pelo nome do medicamento
ORDER BY drugname;

A análise dos cinco medicamentos mais reportados revelou diferenças relevantes no perfil de gravidade dos outcomes associados aos eventos adversos.

O `PREDNISONE` apresentou o maior número de outcomes classificados como Other Serious Outcome (`OT`), bem como valores muito elevados de hospitalização (`HO`) e de morte (`DE`), sugerindo uma associação frequente a eventos adversos graves.

O `ACETAMINOPHEN` destacou-se pelo elevado volume global de reportes, registando também um número considerável de hospitalizações e mortes. Este resultado poderá estar relacionado com a sua ampla utilização na população, aumentando naturalmente o número de notificações recebidas.

O `DUPIXENT` apresentou um perfil aparentemente menos grave quando comparado com os restantes medicamentos analisados, evidenciando um número reduzido de mortes e de eventos classificados como Life-Threatening (`LT`).

O `HUMIRA` revelou uma frequência significativa de hospitalizações e de outcomes graves, nomeadamente eventos Life-Threatening (`LT`) e casos de incapacidade (`DS`), indicando um perfil de segurança que merece atenção especial.

De forma geral, observam-se diferenças importantes entre os medicamentos analisados, particularmente na distribuição dos outcomes mais graves. No entanto, estes resultados devem ser interpretados com cautela, uma vez que refletem notificações espontâneas do sistema FAERS e não estabelecem uma relação causal direta entre o medicamento e o outcome reportado.

# Q7 - Medicamentos com reportes crescentes ao longo do tempo (YoY Growth)

In [0]:
# Reportes por medicamento e ano
reports_by_year = (
    df_gold_base
    .filter(F.col("fda_dt").isNotNull()) # Ignora registos sem data FDA
    .withColumn("report_year", F.year("fda_dt"))
    .groupBy("drugname", "report_year")
    .agg(
        F.countDistinct("primaryid").alias("yearly_reports")
    )
    .filter(F.col("yearly_reports") >= 50) # Ignora medicamentos que tenham poucos casos, para evitar percentagens absurdas
)

# Window para comparar com o ano anterior
w_yoy = (
    Window
    .partitionBy("drugname")
    .orderBy("report_year")
)

q7_result = (
    reports_by_year

    .withColumn(
        "prev_year_reports",
        F.lag("yearly_reports", 1).over(w_yoy)
    )

    .filter(
        F.col("prev_year_reports").isNotNull()
        & (F.col("prev_year_reports") >= 100)
    )

    .withColumn(
        "yoy_growth_pct",  # Fórmula do crescimento YoY(%) // ((ano_atual - ano_anterior) / ano_anterior) x 100
        F.round(
            (
                (F.col("yearly_reports") - F.col("prev_year_reports"))
                / F.col("prev_year_reports")
            ) * 100,
            2
        )
    )

    .orderBy(F.desc("yoy_growth_pct")) # Ordena pelo crescimento YoY

    .select(
        "drugname",
        "report_year",
        "prev_year_reports",
        "yearly_reports",
        "yoy_growth_pct"
    )
    .limit(30) # Mostra apenas as 30 primeiras linhas

)

display(q7_result)

In [0]:
%sql
-- 1. Calcular o número de reportes por medicamento e por ano
WITH reports_by_year AS (

    SELECT
        drugname,
        YEAR(fda_dt) AS report_year, -- Extrair o ano da data de receção pela FDA
        COUNT(DISTINCT primaryid) AS yearly_reports -- Contar casos únicos reportados nesse ano

    FROM gold_base

    WHERE fda_dt IS NOT NULL -- Excluir registos sem data FDA

    GROUP BY
        drugname,
        YEAR(fda_dt)

    HAVING COUNT(DISTINCT primaryid) >= 50 -- Manter apenas combinações medicamento-ano com pelo menos 50 reportes
),

-- 2. Obter o número de reportes do ano anterior
yoy_calculation AS (

    SELECT
        drugname,
        report_year,
        yearly_reports,
-- Obter o valor do ano anterior para o mesmo medicamento
        LAG(yearly_reports, 1) OVER (
            PARTITION BY drugname
            ORDER BY report_year
        ) AS prev_year_reports

    FROM reports_by_year
)
-- 3. Calcular o crescimento Year-over-Year (YoY)
SELECT
    drugname,
    report_year,
    prev_year_reports,
    yearly_reports,

    ROUND(
        (
            (yearly_reports - prev_year_reports)
            / prev_year_reports
        ) * 100,
        2
    ) AS yoy_growth_pct

FROM yoy_calculation
-- Excluir o primeiro ano de cada medicamento (não existe ano anterior para comparação)
WHERE prev_year_reports IS NOT NULL
  AND prev_year_reports >= 100   -- Garantir uma base mínima para evitar crescimentos artificiais devido a valores muito baixos

-- Ordenar pelos maiores crescimentos percentuais
ORDER BY yoy_growth_pct DESC

LIMIT 30; -- Mostrar os 30 maiores aumentos

A análise Year-over-Year (`YoY`) permitiu identificar os medicamentos que registaram os maiores aumentos no número de reportes de eventos adversos entre anos consecutivos. Os resultados evidenciam um crescimento muito expressivo em vários medicamentos, particularmente em 2023.

O medicamento `ACETAMINOPHEN/HYDROCODONE BITARTRATE` apresentou o maior crescimento observado, passando de 187 reportes no ano anterior para 27.470 reportes em 2023, correspondendo a um aumento de aproximadamente 14.590%. Também se destacaram `MINOXIDIL`, `PERCOCET` e `FENTANYL CITRATE`, todos com aumentos superiores a 2.000%, refletindo um crescimento substancial na frequência de notificações.

Outros medicamentos como `TRAMADOL HYDROCHLORIDE`, `VYVGART`, `ULTOMIRIS`, `BOTOX` e `AMPYRA` também apresentaram aumentos relevantes, indicando uma tendência crescente de reportes de eventos adversos ao longo do período analisado.

Estes aumentos podem estar relacionados com diversos fatores, incluindo um maior número de prescrições, expansão das indicações terapêuticas, aumento da utilização clínica, maior sensibilização para a notificação de eventos adversos ou reforço das atividades de farmacovigilância. Assim, um crescimento elevado no número de reportes não implica necessariamente um agravamento do perfil de segurança do medicamento, mas pode indicar a necessidade de monitorização mais detalhada da sua evolução ao longo do tempo.

De forma global, a análise permitiu identificar medicamentos com crescimento acentuado nos reportes de eventos adversos, fornecendo informação útil para a deteção de tendências emergentes e potenciais sinais de segurança na base de dados FAERS.

# Q8 - Construir um "Drug Safety Score"

Foi desenvolvido um Drug Safety Score para sintetizar o perfil de segurança de cada medicamento numa única métrica. O score combina a frequência dos eventos adversos reportados com a gravidade dos outcomes associados, atribuindo pesos superiores a outcomes mais graves, como morte (`DE`) e eventos life-threatening (`LT`). Esta abordagem permite identificar medicamentos com maior impacto global em termos de segurança, considerando simultaneamente o volume de reportes e a severidade clínica dos eventos observados.

In [0]:
# Ranking de medicamentos com maior Drug Safety Score

q8_result = (

    df_gold_base

    # Remove registos sem medicamento ou sem outcome associado
    .filter(
        F.col("drugname").isNotNull()
        & F.col("outc_cod").isNotNull()
    )

    # Agrupa os dados por medicamento
    .groupBy("drugname")

    # Cria uma coluna para cada tipo de outcome
    .pivot(
        "outc_cod",
        ["DE", "LT", "HO", "DS", "RI", "CA", "OT"]
    )

    # Conta o número de casos únicos para cada outcome
    .agg(
        F.countDistinct("primaryid")
    )

    # Substitui valores nulos por zero
    .fillna(0)

    # Calcula o Drug Safety Score com base na gravidade dos outcomes
    .withColumn(
        "drug_safety_score",

        # Outcomes mais graves recebem maior peso
        (
            F.col("DE") * 10 +  # Death
            F.col("LT") * 8 +   # Life-Threatening
            F.col("HO") * 5 +   # Hospitalization
            F.col("DS") * 4 +   # Disability
            F.col("RI") * 3 +   # Required Intervention
            F.col("CA") * 3 +   # Congenital Anomaly
            F.col("OT") * 1     # Other Serious Outcome
        )
    )

    # Ordena os medicamentos pelo score calculado
    .orderBy(
        F.desc("drug_safety_score")
    )

    # Seleciona os 20 medicamentos com maior score
    .limit(20)
)

# Apresenta o ranking dos medicamentos com maior Drug Safety Score
display(q8_result)

In [0]:
%sql
-- 1. Criar uma tabela pivot onde cada outcome passa a ser uma coluna
WITH outcomes_pivot AS (

SELECT *
FROM (
    SELECT
        drugname,
        outc_cod, 
        primaryid
    FROM gold_base
)
-- Transformar os diferentes outcomes em colunas
PIVOT (
    COUNT(DISTINCT primaryid)
    FOR outc_cod IN (
        'DE' AS DE,
        'LT' AS LT,
        'HO' AS HO,
        'DS' AS DS,
        'RI' AS RI,
        'CA' AS CA,
        'OT' AS OT
    )
)

)
-- 2. Calcular o Drug Safety Score para cada medicamento
SELECT
    drugname,
-- Exibir os outcomes mais relevantes (COALESCE - serve para substituir valores NULL por outro valor que for definido (neste caso definimos 0))
    COALESCE(DE,0) AS deaths,
    COALESCE(LT,0) AS life_threatening,
    COALESCE(HO,0) AS hospitalization,
-- Cálculo do score de segurança
    (
        COALESCE(DE,0) * 10 +
        COALESCE(LT,0) * 8 +
        COALESCE(HO,0) * 5 +
        COALESCE(DS,0) * 4 +
        COALESCE(RI,0) * 3 +
        COALESCE(CA,0) * 3 +
        COALESCE(OT,0) * 1
    ) AS drug_safety_score

FROM outcomes_pivot
-- Ordenar os medicamentos pelos scores mais elevados
ORDER BY drug_safety_score DESC
-- Mostrar apenas os 20 medicamentos com maior score
LIMIT 20;

O Drug Safety Score foi construído para combinar simultaneamente a frequência dos reportes e a gravidade dos outcomes associados a cada medicamento. Quanto maior o score, maior o impacto global do medicamento em termos de segurança dentro da base de dados FAERS.

Os resultados mostram que o `ACETAMINOPHEN` obteve o maior Drug Safety Score (302.716), refletindo uma combinação de elevado número de reportes, hospitalizações, eventos life-threatening e mortes. Este resultado sugere um impacto muito significativo na base de farmacovigilância analisada.

O `PREDNISONE` surge na segunda posição (291.881), destacando-se sobretudo pelo elevado número de outcomes classificados como Other Serious Outcome (`OT`) e pelo grande volume de hospitalizações e mortes associadas.

O `ASPIRIN` ocupa o terceiro lugar (211.862), mantendo um perfil de segurança relevante devido ao elevado número de hospitalizações e outcomes graves reportados.

Medicamentos como `DEXAMETHASONE`, `METHOTREXATE` e `RITUXIMAB` também apresentam scores elevados, evidenciando uma frequência significativa de eventos adversos graves e outcomes de elevada severidade.

De forma geral, observa-se que os medicamentos com maior Drug Safety Score são aqueles que combinam:

- Um elevado volume de reportes;
- Um número significativo de hospitalizações;
- Uma frequência relevante de eventos life-threatening;
- Um número elevado de mortes reportadas.

# Persistência da Camada Gold

Após a execução de todas as análises (Q1-Q8), a tabela `gold_base` é persistida em formato Delta para permitir reutilização futura.

**Estratégia de Persistência:**
* Durante as análises, a tabela `df_gold_base` é mantida em memória para máximo desempenho
* No final, as colunas duplicadas (`load_timestamp`, `source_file`) são removidas antes de guardar!
* A tabela é guardada em formato Delta para persistência e reutilização

**Por que guardar no final?**
* ✅ **Desempenho**: Leitura de memória é mais rápida que Delta (sem I/O, sem deserialização)
* ✅ **Flexibilidade**: Todas as análises usam a mesma fonte em memória
* ✅ **Persistência**: Delta permite reutilização em futuras sessões

In [0]:
# Definir o caminho para guardar a tabela Gold
gold_delta_path = "/Volumes/main/default/faers_data/delta/gold"

print("="*80)
print("💾 GUARDAR GOLD BASE TABLE EM DELTA")
print("="*80 + "\n")

# Remover colunas duplicadas apenas para o save (mantém df_gold_base intacto)
metadata_cols = ["load_timestamp", "source_file"]

print("🗑️  Removendo colunas duplicadas para o save...")
print(f"   Colunas a remover: {metadata_cols}\n")

df_gold_to_save = df_gold_base.drop(*metadata_cols)

print("💾 Guardando tabela em formato Delta...")
df_gold_to_save.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_delta_path}/gold_base")

print(f"\n✅ Tabela gold_base guardada em: {gold_delta_path}/gold_base")
print("\n📈 BENEFÍCIOS:")
print("   • Persistência: Tabela guardada em disco para reutilização")
print("   • Delta Features: ACID transactions, time travel, schema enforcement")
print("   • No duplicates: Colunas load_timestamp e source_file removidas")
print("\n📁 Leitura da tabela:")
print(f"   spark.read.format('delta').load('{gold_delta_path}/gold_base')")